# 02 - 语雀文档 CRUD 完整验证

> 测试目标：验证语雀内部 Web API 的 CREATE → READ → UPDATE → READ → DELETE 完整流程。
> 核心发现：**必须使用 `body_asl` 字段保存内容**，使用 `body` 会导致内容为空。

## 测试目录

1. [环境准备](#环境准备) — 加载 Cookie，初始化客户端
2. [创建文档](#创建文档) — POST `/api/docs`，使用 `body_asl`
3. [读取验证](#读取验证) — GET `/api/docs/{slug}`，验证内容不为空
4. [更新文档](#更新文档) — PUT `/api/docs/{id}`，修改标题和内容
5. [再次读取验证](#再次读取验证) — 确认更新生效
6. [删除文档](#删除文档) — DELETE `/api/docs/{id}`

---

## 环境准备

In [ ]:
# =============================================================================
# 环境准备：加载依赖、初始化客户端
# =============================================================================

import os
import json
from pathlib import Path
from datetime import datetime

# 加载 .env（如果编码有问题则手动填写）
try:
    from dotenv import load_dotenv
    load_dotenv(Path('../../.env'))
except ImportError:
    pass

from yuque_client import YuqueClient, lake_html

# 初始化客户端
client = YuqueClient()
print('✅ 客户端初始化成功')

# 获取测试用的知识库
books = client.list_books()
BOOK_ID = books[0]['id']
BOOK_SLUG = books[0]['slug']
print(f'📚 使用知识库: {books[0]["name"]} (ID={BOOK_ID})')

# 全局变量：记录测试创建的文档，用于最后清理
TEST_DOC_IDS = []

---

## 创建文档

**端点**：`POST /api/docs`

**请求体**：
```json
{
  "book_id": 68025057,           // 知识库 ID（数字）
  "title": "文档标题",            // 文档标题
  "body_asl": "<!doctype lake>...",  // 内容（Lake HTML）【必须用 body_asl！】
  "format": "lake",              // 格式：lake（固定）
  "public": 0                    // 可见性：0=私密
}
```

**⚠️ 关键提醒**：
- **必须使用 `body_asl` 字段**，使用 `body` 会导致 API 返回成功但内容为空（0 chars）
- 写操作必须有 `Referer` 头，否则返回 403
- 创建后文档不会自动出现在目录中

In [ ]:
# =============================================================================
# 创建文档
# =============================================================================

# 构建 Lake HTML 内容
content = lake_html(
    '<h1>CRUD 测试文档</h1>',
    '<p>这是一段普通文本，用于验证内容是否正确保存。</p>',
    '<ul><li>列表项 1</li><li>列表项 2</li></ul>',
)

doc = client.create_doc(
    book_id=BOOK_ID,
    title=f'[CRUD]测试文档-{datetime.now().strftime("%H%M%S")}',
    content=content,
)

if not doc:
    raise RuntimeError('创建文档失败')

DOC_ID = doc['id']
DOC_SLUG = doc['slug']
TEST_DOC_IDS.append(DOC_ID)

print(f'✅ 创建成功')
print(f'   ID: {DOC_ID}')
print(f'   Slug: {DOC_SLUG}')
print(f'   标题: {doc["title"]}')
print(f'   URL: https://www.yuque.com/{BOOK_SLUG}/{DOC_SLUG}')

---

## 读取验证

**端点**：`GET /api/docs/{slug}?book_id={id}`

**关键检查点**：
1. `content` 字段长度是否大于 0
2. 内容中是否包含我们写入的文本

**如果这一步失败，说明 `body_asl` 字段没有正确生效。**

In [ ]:
# =============================================================================
# 读取文档并验证内容
# =============================================================================

doc = client.read_doc(BOOK_ID, DOC_SLUG)

print(f'📄 文档详情:')
print(f'   标题: {doc.get("title")}')
print(f'   ID: {doc.get("id")}')
print(f'   格式: {doc.get("format")}')
print(f'   字数: {doc.get("word_count")}')

content = doc.get('content', '')
print(f'\n📋 内容长度: {len(content)} 字符')
print(f'\n--- 内容预览（前 400 字符）---')
print(content[:400])

# 关键验证
checks = [
    (len(content) > 0,     'content 长度 > 0'),
    ('CRUD 测试文档' in content,   '包含标题文字'),
    ('验证内容是否正确保存' in content, '包含正文文字'),
    ('列表项 1' in content,        '包含列表项'),
]

all_pass = True
for passed, desc in checks:
    status = '✅' if passed else '❌'
    print(f'\n   {status} {desc}')
    if not passed:
        all_pass = False

if all_pass:
    print('\n✅ ✅ ✅ 内容验证通过！body_asl 字段工作正常！')
else:
    print('\n❌ ❌ ❌ 内容验证失败！请检查 body_asl 字段是否正确使用。')

---

## 更新文档

**端点**：`PUT /api/docs/{id}`

**请求体**：
```json
{
  "title": "新标题",               // 新标题（可选）
  "body_asl": "<!doctype lake>...",  // 新内容（可选）【必须用 body_asl！】
  "format": "lake"                // 格式（固定 lake）
}
```

**⚠️ 重要提醒**：
- URL 中的 `{id}` 是**数字 ID**（如 `266476793`），不是 slug！
- 必须先调用 read 获取 doc_id
- 写操作必须有 `Referer` 头

In [ ]:
# =============================================================================
# 更新文档：修改标题和内容
# =============================================================================

new_content = lake_html(
    '<h1>CRUD 测试文档 — 已更新</h1>',
    '<p>这是更新后的内容，添加了<strong>加粗文字</strong>和<em>斜体文字</em>。</p>',
    '<ol><li>有序项 A</li><li>有序项 B</li></ol>',
)

updated = client.update_doc(
    doc_id=DOC_ID,
    title=f'[CRUD]测试文档-已更新-{datetime.now().strftime("%H%M%S")}',
    content=new_content,
)

if updated:
    print(f'✅ 更新成功')
    print(f'   新标题: {updated.get("title")}')
    print(f'   更新时间: {updated.get("updated_at", "N/A")}')
else:
    print(f'❌ 更新失败')

---

## 再次读取验证

再次读取文档，确认：
1. 标题已更新
2. 内容已更新（包含新写入的文本）
3. 旧内容被替换（不再包含旧的列表项）

In [ ]:
# =============================================================================
# 再次读取验证更新是否生效
# =============================================================================

doc = client.read_doc(BOOK_ID, DOC_SLUG)
content = doc.get('content', '')

print(f'📄 再次读取: {doc.get("title")}')
print(f'📋 内容长度: {len(content)} 字符')
print(f'\n--- 内容预览（前 400 字符）---')
print(content[:400])

# 验证更新
checks = [
    ('已更新' in content,           '包含新标题文字'),
    ('更新后的内容' in content,     '包含新正文'),
    ('加粗文字' in content,         '包含加粗文字'),
    ('有序项 A' in content,         '包含新列表项'),
    ('列表项 1' not in content,     '旧列表项已移除'),
]

all_pass = True
for passed, desc in checks:
    status = '✅' if passed else '❌'
    print(f'\n   {status} {desc}')
    if not passed:
        all_pass = False

if all_pass:
    print('\n✅ ✅ ✅ 更新验证通过！')
else:
    print('\n❌ ❌ ❌ 更新验证失败！')

---

## 删除文档

**端点**：`DELETE /api/docs/{id}?book_id={book_id}`

**⚠️ 警告**：删除操作不可逆！

In [ ]:
# =============================================================================
# 删除测试文档
# =============================================================================

print(f'准备删除文档: ID={DOC_ID}')

success = client.delete_doc(DOC_ID, BOOK_ID)
if success:
    print('✅ 删除成功')
    TEST_DOC_IDS.remove(DOC_ID)
else:
    print('❌ 删除失败')

---

## 附录：body vs body_asl 对比实测

以下代码展示使用 `body` 和 `body_asl` 的区别：

In [ ]:
# =============================================================================
# 对比测试：body vs body_asl
# =============================================================================

import requests

def raw_api(method, path, data=None, query=None, referer=None):
    """直接调用 API，便于测试不同字段"""
    url = f'{client.BASE_URL}{path}'
    if query:
        url += '?' + '&'.join(f'{k}={v}' for k, v in query.items())
    h = {
        'Cookie': client.cookie,
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'application/json, text/plain, */*',
        'X-CSRF-Token': client.ctoken,
        'X-Requested-With': 'XMLHttpRequest',
    }
    if referer:
        h['Referer'] = referer
    if data and method not in ('GET', 'DELETE'):
        h['Content-Type'] = 'application/json'
    if method == 'POST':
        r = requests.post(url, headers=h, json=data)
    return r.json()

test_body = '<!doctype lake><h1>字段对比测试</h1><p>测试不同字段的效果。</p>'

print('--- 使用 body 字段 ---')
r1 = raw_api('POST', '/api/docs', data={
    'book_id': BOOK_ID,
    'title': f'[字段对比]body-{datetime.now().strftime("%H%M%S")}',
    'body': test_body,
    'format': 'lake',
}, referer=f'{client.BASE_URL}/{BOOK_ID}')

if 'data' in r1:
    slug1 = r1['data']['slug']
    doc1 = client.read_doc(BOOK_ID, slug1)
    len1 = len(doc1.get('content', ''))
    print(f'   content 长度: {len1} chars → {"✅ 有内容" if len1 > 0 else "❌ 为空"}')
    raw_api('DELETE', f'/api/docs/{r1["data"]["id"]}', query={'book_id': str(BOOK_ID)}, referer=f'{client.BASE_URL}/{BOOK_ID}')

print('\n--- 使用 body_asl 字段 ---')
r2 = raw_api('POST', '/api/docs', data={
    'book_id': BOOK_ID,
    'title': f'[字段对比]body_asl-{datetime.now().strftime("%H%M%S")}',
    'body_asl': test_body,
    'format': 'lake',
}, referer=f'{client.BASE_URL}/{BOOK_ID}')

if 'data' in r2:
    slug2 = r2['data']['slug']
    doc2 = client.read_doc(BOOK_ID, slug2)
    len2 = len(doc2.get('content', ''))
    print(f'   content 长度: {len2} chars → {"✅ 有内容" if len2 > 0 else "❌ 为空"}')
    raw_api('DELETE', f'/api/docs/{r2["data"]["id"]}', query={'book_id': str(BOOK_ID)}, referer=f'{client.BASE_URL}/{BOOK_ID}')

print('\n📌 结论：body_asl 才是正确的内容字段！')